# Aspect-Based Sentiment Analysis for Restaurant Ranking

## Load Data

In [21]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Read raw data
master_dir = os.path.dirname(os.getcwd())

file_google = os.path.join(master_dir, "data", "raw_data", "GoogleReview_data.csv")
file_tripadvisor = os.path.join(master_dir, "data", "raw_data", "TripAdvisor_data.csv")

df_google = pd.read_csv(file_google)
df_tripadvisor = pd.read_csv(file_tripadvisor)

## Preprocessing

In [22]:
df_google.count()

Author        222020
Rating        222020
Review        222020
Restaurant    222020
Location      222020
dtype: int64

In [23]:
df_tripadvisor.count()

Author        139764
Title         139764
Review        139764
Rating        139764
Dates         139764
Restaurant    139764
Location      139764
dtype: int64

In [24]:
df = pd.concat([df_google, df_tripadvisor], ignore_index=True)

df.count()

Author        361784
Rating        361784
Review        361784
Restaurant    361784
Location      361784
Title         139764
Dates         139764
dtype: int64

### Basic Cleaning

In [ ]:
import emoji
import re
import contractions

def basic_cleaning(df, keep_emoji_text=True):
    # Remove duplicates
    df = df.drop_duplicates()

    #remomve rows with empty or NaN reviews
    df = df[
        df["Review"].notna() &
        df["Review"].astype(str).str.strip().ne("")
    ]
    
    # Remove HTML tags
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'<[^>]+>', '', x))

    # Remove URLs
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'http\S+|www\S+', '[URL]', x))

    # Remove email addresses
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'\S+@\S+', '[EMAIL]', x))

    # Normalize whitespace
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

    # Keep or remove emojis based on the parameter 
    if keep_emoji_text:
        df["Review"] = df["Review"].apply(lambda x: emoji.demojize(x, delimiters=(' ', ' ')))
    else:
        df["Review"] = df["Review"].apply(lambda x: emoji.replace_emoji(x, replace=''))

    # Expand contractions
    df["Review"] = df["Review"].apply(lambda x: contractions.fix(x))

    # Remove mentions
    df["Review"] = df["Review"].apply(lambda x: re.sub(r'@\w+', '', x))

    return df

In [35]:
# clean the dataframes
df_cleaned = basic_cleaning(df)
df_google_cleaned = basic_cleaning(df_google)
df_tripadvisor_cleaned = basic_cleaning(df_tripadvisor)

In [38]:
# number of records
sampleNum = df_cleaned.index.size
print(f"Sample Number: {sampleNum}")

Sample Number: 361779


In [39]:
# subset wanted columns
def subset_columns(df):
    df = df[["Review", "Rating"]]
    return df  

In [40]:
# remain only the wanted columns
df_cleaned = subset_columns(df_cleaned)
df_google_cleaned = subset_columns(df_google_cleaned)
df_tripadvisor_cleaned = subset_columns(df_tripadvisor_cleaned)

In [41]:
def count_caps_words(text):
    words = re.findall(r'\b[A-Za-z]+\b', text)
    
    return sum(
        1 for word in words
        if len(word) > 1 and word.isupper()
    )

In [ ]:
df_cleaned["caps_count"] = df["Review"].apply(count_caps_words)
df_google_cleaned["caps_count"] = df_google_cleaned["Review"].apply(count_caps_words)
df_tripadvisor_cleaned["caps_count"] = df_tripadvisor_cleaned["Review"].apply(count_caps_words)

df_cleaned.head(10)

,Review,Rating,caps_count
0,came here for the high tea. great service espe...,4.0,0
1,"5 stars for the service, even though some of t...",2.0,0
2,"hi, thank you for your service. but! i feel so...",1.0,0
3,i have the worse buffer dinner ever so far. th...,1.0,0
4,"that is are known 5 elmark "" [CAPS]9h72[/CAPS]...",5.0,1
5,i just came back from there. 2 adults and 4 yo...,2.0,0
6,restaurant looks nice but taste is bad. i had ...,2.0,1
7,"pros: ambience is great with lake view, good a...",4.0,0
8,we went to this place after reviews on tripadv...,1.0,0
9,"the restaurant is located inside the hotel, th...",4.0,0


### Tokenization

In [46]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

def tokenize_review(text):
    # Sentence tokenization
    sentences = sent_tokenize(text)
    
    # Word tokenization
    tokens = []
    for sentence in sentences:
        tokens.extend(word_tokenize(sentence))
    
    return sentences, tokens

In [47]:
df_cleaned[["Sentences", "Tokens"]] = df["Review"].apply(
    lambda x: pd.Series(tokenize_review(x))
)
df_google_cleaned[["Sentences", "Tokens"]] = df_google_cleaned["Review"].apply(
    lambda x: pd.Series(tokenize_review(x))
)
df_tripadvisor_cleaned[["Sentences", "Tokens"]] = df_tripadvisor_cleaned["Review"].apply(
    lambda x: pd.Series(tokenize_review(x))
)

### Stop-word removal

In [49]:
import nltk
from nltk.corpus import stopwords

# Base English stopword list
stop_words = set(stopwords.words("english"))

# Words that carry sentiment, negation, or contrast
important_words = {
    "not",
    "no",
    "nor",
    "never",
    "neither",
    "none",
    "but",
    "however",
    "although",
    "yet",
    "though",
    "very",
    "too",
    "so",
    "really",
    "extremely",
    "quite",
    "highly",
    "somewhat",
    "slightly",
    "badly",
    "hardly",
    "absolutely",
    "completely",
    "could",
    "would",
    "should",
    "might",
    "must"
}

# Keep important words by removing them from stopword list
stop_words = stop_words - important_words

def remove_stopwords(tokens):
    return [
        token for token in tokens
        if token.lower() not in stop_words
    ]

In [50]:
df_cleaned["Tokens_no_stopwords"] = df_cleaned["Tokens"].apply(remove_stopwords)
df_google_cleaned["Tokens_no_stopwords"] = df_google_cleaned["Tokens"].apply(remove_stopwords)
df_tripadvisor_cleaned["Tokens_no_stopwords"] = df_tripadvisor_cleaned["Tokens"].apply(remove_stopwords)

### Lemmatization